# Adapted from https://milvus.io/docs/multi-vector-search.md

In [1]:
%%capture
from pymilvus import MilvusClient, DataType, Function, FunctionType, AnnSearchRequest, RRFRanker

client = MilvusClient("./milvus_demo.db")

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('multi-qa-mpnet-base-cos-v1')

I0814 09:53:15.156887  223821 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0814 09:53:15.176089  224186 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0814 09:53:15.176146  224186 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0814 09:53:15.176148  224186 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(93, generation: 1)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
schema = client.create_schema()

schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=768)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [4]:
bm25_function = Function(
    name="text_bm25_emb", # Function name
    input_field_names=["text"], # Name of the VARCHAR field containing raw text data
    output_field_names=["sparse"], # Name of the SPARSE_FLOAT_VECTOR field reserved to store generated embeddings
    function_type=FunctionType.BM25, # Set to `BM25`
)

schema.add_function(bm25_function)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_output': True}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'text_bm25_emb', 'description': '', 'type': <FunctionType.BM25: 1>, 'input_field_names': ['text'], 'output_field_names': ['sparse'], 'params': {}}]}

In [5]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.2,
        "bm25_b": 0.75
    }
)

index_params.add_index(
    field_name="dense",
    index_name="text_dense_index",
    index_type="AUTOINDEX",
    metric_type="IP"
)

In [6]:
# Drop existing collection with the same name if it exists
if client.has_collection("nyu_rt_docs"):
    client.drop_collection("nyu_rt_docs")

In [7]:
client.create_collection(
    collection_name='nyu_rt_docs', 
    schema=schema, 
    index_params=index_params
)

## Set up chunker for use

In [8]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

converter = DocumentConverter()
chunker = HybridChunker()

In [9]:
import pickle

with open("docs.pickle", "rb") as f:
    docs = pickle.load(f)

In [10]:
for subjects in docs['https://services.rt.nyu.edu/docs/hpc/getting_started/intro/']:
    print(subjects)

https://services.rt.nyu.edu/docs/hpc/getting_started/intro/


In [ ]:
from tqdm import tqdm

#for guide in tqdm(["https://services.rt.nyu.edu/docs/hpc/getting_started/intro/"]):
for guide in tqdm(docs.keys()):
    for subject in docs[guide]:
        DOC_SOURCE = subject
        try:
            doc = converter.convert(source=DOC_SOURCE).document
            chunk_iter = chunker.chunk(dl_doc=doc)
            texts = [chunk.text for chunk in chunker.chunk(doc)]
    
            for text in texts:
                client.insert('nyu_rt_docs', [{'text': text, 'dense': model.encode(text)}])
        except:
            pass

  0%|                                                                                         | 0/121 [00:00<?, ?it/s]

In [ ]:
query = "How do I request an HPC account?"

search_params_sparse = {
    "data": [query],
    "anns_field": "sparse",
    "param": {"drop_ratio_search": 0.2},
    "limit": 5,

}
sparse_request = AnnSearchRequest(**search_params_sparse)

search_params_dense = {
    "data": [model.encode(query)],
    "anns_field": "dense",
    "param": {"nprobe": 10},
    "limit": 5
}
dense_request = AnnSearchRequest(**search_params_dense)

In [ ]:
reqs = [sparse_request, dense_request]
ranker = RRFRanker()

res = client.hybrid_search(
    collection_name="nyu_rt_docs",
    reqs=reqs,
    ranker=ranker,
    limit=3,
    output_fields=["text"],  # Return id and species
)
for hits in res:
    print("Hybrid Search results:")
    for hit in hits:
        print("-----------------------------------------")
        print(f"{hit.entity.get('text')}")